# Parte 1 — Ingestão e Preparação (Camadas Bronze)

###  1.1. Preparação e Upload dos Dados
Carregue os arquivos CSV disponibilizados na pasta Inputs como Volume no Data Lakehouse do Databricks.


In [0]:
# Importações e criação da sessão Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Criar sessão spark
spark = SparkSession.builder.appName("AtividadePraticaMedallion").getOrCreate()

In [0]:
# Configuração de catálogo, schemas e caminho da landing zone
catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"Landing: {landing_path}")

catalog: olist_medallion
bronze_schema: olist_medallion.bronze
silver_schema: olist_medallion.silver
gold_schema: olist_medallion.gold
Landing: /Volumes/olist_medallion/bronze/landing


%md
### 1.2. Notebook — Camada Bronze

In [0]:
# Criação do catálogo, schemas e volume da landing zone
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

DataFrame[]

In [0]:
# Validação dos arquivos esperados na landing zone
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar {landing_path}. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos estão na landing zone.")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos estão na landing zone.
  - cotacao_dolar.csv (0.2 KB)
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)


### Ingestão Bronze

In [0]:
# Leitura dos CSVs da landing zone para DataFrames da camada Bronze
from pyspark.sql.functions import current_timestamp

path_credits_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

df_bronze_credits_tags_raw = spark.read.csv(path_credits_tags, header=True, inferSchema=True)
df_bronze_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=True)
df_bronze_movies_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=True)
df_bronze_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=True)
df_bronze_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=True)

In [0]:
# Persistência das tabelas Bronze com timestamp de ingestão
(
    df_bronze_credits_tags_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.credits")
)

(
    df_bronze_movies_financials_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.movies_financials")
)

(
    df_bronze_movies_info_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.movies_info")
)

(
    df_bronze_movies_metrics_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.movies_metrics")
)

(
    df_bronze_movies_reviews_raw
    .withColumn("ingestion_datetime", current_timestamp())
    .write.format("Delta").mode("append")
    .saveAsTable(f"{catalog}.bronze.movies_reviews")
)

display(spark.table(f"{catalog}.bronze.credits").limit(5))

id,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,cast,ingestion_datetime
293660,"Action, Adventure, Comedy","20th Century Fox, The Donners' Company, Genre Films",United States of America,English,"superhero, anti hero, mercenary, based on comic, aftercreditsstinger, duringcreditsstinger",Tim Miller,"Rhett Reese, Paul Wernick","Ryan Reynolds, Morena Baccarin, Ed Skrein, T.J. Miller, Gina Carano, Leslie Uggams, Brianna Hildebrand, Stefan Kapičić, Karan Soni, Randal Reeder",2026-10-12T22:48:46.533Z
299536,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"English, Xhosa","sacrifice, magic, superhero, based on comic, space, battlefield, genocide, magical object, super power, aftercreditsstinger, marvel cinematic universe (mcu), cosmic","Anthony Russo, Joe Russo",N/A,"Robert Downey Jr., Chris Evans, Chris Hemsworth, Josh Brolin, Mark Ruffalo, Scarlett Johansson, Don Cheadle, Benedict Cumberbatch, Tom Holland, Chadwick Boseman",2026-10-12T22:48:46.533Z
299534,"Adventure, Science Fiction, Action",Marvel Studios,United States of America,"English, Japanese, Xhosa","superhero, time travel, space travel, time machine, based on comic, sequel, alien invasion, superhero team, marvel cinematic universe (mcu), alternate timeline, father daughter relationship, sister sister relationship","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Stan Lee, Jack Kirby, Joe Simon, Steve Englehart, Steve Gan, Bill Mantlo, Keith Giffen, Jim Starlin, Larry Lieber, Don Heck","Robert Downey Jr., Chris Evans, Mark Ruffalo, Chris Hemsworth, Scarlett Johansson, Jeremy Renner, Josh Brolin, Don Cheadle, Paul Rudd, Benedict Cumberbatch",2026-10-12T22:48:46.533Z
475557,"Crime, Thriller, Drama","Warner Bros. Pictures, Joint Effort, Village Roadshow Pictures, Bron Studios, DC Films","Canada, United States of America",English,"dream, street gang, society, psychopath, clown, villain, based on comic, murder, psychological thriller, criminal mastermind, mental illness, anarchy, character study, clown makeup, subway train, social realism, supervillain, tv host, 1980s, mother son relationship, origin story, falling into madness, depressing",Todd Phillips,"Todd Phillips, Scott Silver, Bob Kane, Bill Finger, Jerry Robinson","Joaquin Phoenix, Robert De Niro, Zazie Beetz, Frances Conroy, Brett Cullen, Shea Whigham, Bill Camp, Glenn Fleshler, Leigh Gill, Josh Pais",2026-10-12T22:48:46.533Z
271110,"Adventure, Action, Science Fiction",Marvel Studios,United States of America,"Romanian, English, German, Russian","civil war, superhero, based on comic, sequel, aftercreditsstinger, duringcreditsstinger, marvel cinematic universe (mcu), excited","Anthony Russo, Joe Russo","Christopher Markus, Stephen McFeely, Joe Simon, Jack Kirby","Chris Evans, Robert Downey Jr., Scarlett Johansson, Sebastian Stan, Anthony Mackie, Don Cheadle, Jeremy Renner, Chadwick Boseman, Paul Bettany, Elizabeth Olsen",2026-10-12T22:48:46.533Z


### Ingestão de API

As próximas duas células de código que realizam a ingestão da API foram feitas com o auxílio de algoritmos compartilhados no grupo de Whatapp do RocketLab.

OBS: A princípio, os dados de cotação do dólar seriam obtidos diretamente pela API do Banco Central do Brasil (Olinda/SGS). Porém, devido a restrições do ambiente do Databricks, não foi possível realizar a consulta à API diretamente. Para dar continuidade ao desenvolvimento do projeto, foi utilizado um mockup, com os dados da cotação obtidos previamente e salvos em um arquivo CSV. Dessa forma, o arquivo cotacao_dolar.csv simula o resultado que seria retornado pela API e permite executar as etapas seguintes do código normalmente. O código foi modificado, de maneira que ao tentar buscar os dados pela API e falhar, o algoritmo fará a leitura a partir do mockup. 


In [0]:
# Widgets para definir o intervalo de datas da cotação do dólar
from datetime import date, timedelta

dbutils.widgets.text("data_inicio", (date.today() - timedelta(days=7)).strftime("%m-%d-%Y"), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", date.today().strftime("%m-%d-%Y"), "Data fim (MM-DD-AAAA)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

In [0]:
# Ingestão da cotação do dólar: tentativa via API do BCB com fallback para CSV
import requests
import pandas as pd

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

try:
    resposta = requests.get(url, timeout=30)
    resposta.raise_for_status()
    cotacoes = resposta.json()["value"]

    if not cotacoes:
        raise ValueError("A API não retornou cotações para o período informado. Ajuste as datas nos widgets.")

    df_cotacao_raw = spark.createDataFrame(pd.DataFrame(cotacoes))
    print("[OK] Cotações obtidas via API do BCB.")

except Exception as e:
    print(f"[AVISO] Falha ao consultar a API do BCB: {type(e).__name__}: {e}")
    print("Tentando carregar cotações a partir de arquivo CSV na landing zone...")

    csv_path = f"{landing_path}/cotacao_dolar.csv"
    try:
        df_cotacao_raw = spark.read.csv(csv_path, header=True, inferSchema=True)
        print(f"[OK] Cotações carregadas a partir de {csv_path}")
    except Exception as csv_e:
        raise RuntimeError(
            f"Não foi possível obter as cotações do dólar.\n"
            f"  - API do BCB falhou (sem acesso à internet no compute serverless).\n"
            f"  - Arquivo CSV não encontrado em {csv_path}.\n"
            f"Solução: baixe manualmente as cotações no site olinda.bcb.gov.br, "
            f"salve como 'cotacao_dolar.csv' e faça upload para a landing zone, "
            f"ou utilize um cluster clássico com acesso à internet."
        ) from csv_e

df_cotacao_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")

# spark.sql(f"TRUNCATE TABLE {bronze_schema}.tb_cotacao_dolar")

# display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").orderBy("dataHoraCotacao"))

display(
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .select(
        F.date_format("dataHoraCotacao", "MM-dd-yyyy").alias("dataHoraCotacao"),
        "cotacaoCompra",
        "ingestion_datetime"
    )
    .orderBy("dataHoraCotacao")
)

[AVISO] Falha ao consultar a API do BCB: ConnectionError: HTTPSConnectionPool(host='olinda.bcb.gov.br', port=443): Max retries exceeded with url: /olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='09-30-2026'&@dataFinalCotacao='10-07-2026'&$select=dataHoraCotacao,cotacaoCompra&$format=json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0xff8841255730>: Failed to resolve 'olinda.bcb.gov.br' ([Errno -3] Temporary failure in name resolution)"))
Tentando carregar cotações a partir de arquivo CSV na landing zone...
[OK] Cotações carregadas a partir de /Volumes/olist_medallion/bronze/landing/cotacao_dolar.csv


dataHoraCotacao,cotacaoCompra,ingestion_datetime
09-30-2026,5.1803,2026-10-07T21:29:58.497Z
10-01-2026,5.2073,2026-10-07T21:29:58.497Z
10-02-2026,5.2232,2026-10-07T21:29:58.497Z
10-05-2026,4.9853,2026-10-07T21:29:58.497Z
10-06-2026,4.9692,2026-10-07T21:29:58.497Z
